In [ ]:
import numpy as np
import pandas as pd
import os


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.csv'):
            print(os.path.join(dirname, filename))


csv_path = '/kaggle/input/layoutlm/medquad.csv'

try:
    df = pd.read_csv(csv_path)
    print("Data loaded successfully!")
    print(f"Total Questions: {len(df)}")
    display(df.head())
except FileNotFoundError:
    print("Could not find file. Please check the path printed above.")

In [ ]:
import pandas as pd
import numpy as np
csv_path = '/kaggle/input/layoutlm/medquad.csv'

try:
    df = pd.read_csv(csv_path)
    print("Data loaded successfully!")
    print(f"Total Questions: {len(df)}")
    display(df.head())
except FileNotFoundError:
    print("Could not find file. Please check the path printed above.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# the distribution of medical focus areas (Diseases vs Drugs vs Other)
plt.figure(figsize=(10,6))
df['focus_area'].value_counts().head(20).plot(kind='bar')
plt.title('Top 20 Medical Focus Areas')
plt.xlabel('Condition / Topic')
plt.ylabel('Count')

plt.savefig("focus_area_plot.png", dpi=300, bbox_inches="tight")

# Show the figure
plt.show()

print("Plot saved as focus_area_plot.png")

# Print an example Question and Answer
print(f"Sample Question: {df.iloc[0]['question']}")
print(f"Sample Answer: {df.iloc[0]['answer']}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# the distribution of medical focus areas
plt.figure(figsize=(10,6))
df['focus_area'].value_counts().head(20).plot(kind='bar')
plt.title('Top 20 Medical Focus Areas')
plt.xlabel('Condition / Topic')
plt.ylabel('Count')

# Save the plot for download
plt.savefig("/kaggle/working/focus_area_plot.png", dpi=300, bbox_inches="tight")

plt.show()

print("Image saved at /kaggle/working/focus_area_plot.png")


In [ ]:

!pip install -q sentence-transformers

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


try:
   
    df = pd.read_csv('/kaggle/input/layoutlm/medquad.csv')
    
    # Clean the data
    df = df.dropna(subset=['question', 'answer'])
    df = df.drop_duplicates(subset=['question']) # Remove duplicate questions
    
    
    df = df.reset_index(drop=True)
    
   
    df_sample = df 
    
    print(f"✅ Full Data Loaded: {len(df_sample)} rows ready.")
except FileNotFoundError:
    print("❌ Error: File not found. Try: /kaggle/input/healthcare-nlp-llms-transformers-datasets/medquad.csv")


print("⏳ Loading Models... (This takes about 30-60 seconds)")

model_general = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ General Model Loaded!")

model_medical = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')
print("✅ Medical Model Loaded!")

In [ ]:
from sentence_transformers import util
import time


questions = df_sample['question'].tolist()

print(f"📚 Indexing {len(questions)} medical questions...")

# --- 1. Index with General Model ---
print("\n🤖 General Model: Reading all questions...")
start = time.time()
embeddings_general = model_general.encode(questions, batch_size=64, show_progress_bar=True, convert_to_tensor=True)
print(f"✅ General Index Created! (Time: {time.time()-start:.1f}s)")

# --- 2. Index with Medical Model ---
print("\n👨‍⚕️ Medical Model: Reading all questions...")
start = time.time()
embeddings_medical = model_medical.encode(questions, batch_size=64, show_progress_bar=True, convert_to_tensor=True)
print(f"✅ Medical Index Created! (Time: {time.time()-start:.1f}s)")

print("\n🎉 SYSTEM READY: You can now ask questions!")

In [ ]:
def ask_doctor(user_query):
 
    query_emb_gen = model_general.encode(user_query, convert_to_tensor=True)
    query_emb_med = model_medical.encode(user_query, convert_to_tensor=True)
    
   
    hit_gen = util.semantic_search(query_emb_gen, embeddings_general, top_k=1)[0][0]
    hit_med = util.semantic_search(query_emb_med, embeddings_medical, top_k=1)[0][0]
    
  
    q_gen = df_sample.iloc[hit_gen['corpus_id']]['question']
    ans_gen = df_sample.iloc[hit_gen['corpus_id']]['answer']
    
    
    q_med = df_sample.iloc[hit_med['corpus_id']]['question']
    ans_med = df_sample.iloc[hit_med['corpus_id']]['answer']
    
   
    print(f"🔎 YOUR QUERY: {user_query}")
    print("="*60)
    
    print(f"🤖 GENERAL AI (Confidence: {hit_gen['score']:.3f})")
    print(f"   Matched Question: {q_gen}")
    print(f"   Answer: {str(ans_gen)[:150]}...") # Show first 150 chars
    print("-" * 60)
    
    print(f"👨‍⚕️ MEDICAL AI (Confidence: {hit_med['score']:.3f})")
    print(f"   Matched Question: {q_med}")
    print(f"   Answer: {str(ans_med)[:150]}...")
    print("="*60)


ask_doctor("What are the signs of renal failure?")

In [ ]:
# Test Case 1: Simple
compare_models("What causes high blood pressure?")

# Test Case 2: Medical Terminology

compare_models("Treatment for pulmonary hypertension")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


test_queries = [
    "Alopecia causes",           # Common term for Hair Loss
    "Myocardial infarction",     # Technical term for Heart Attack
    "Hypertension symptoms",     # High Blood Pressure
    "Carcinoma treatment",       # Cancer
    "Renal failure signs"        # Kidney failure
]


results = []

print("Running benchmark test...")
for query in test_queries:
  
    q_gen = model_general.encode(query, convert_to_tensor=True)
    q_med = model_medical.encode(query, convert_to_tensor=True)
    
   
    hit_gen = util.semantic_search(q_gen, embeddings_general, top_k=1)[0][0]
    hit_med = util.semantic_search(q_med, embeddings_medical, top_k=1)[0][0]
    
 
    results.append({
        "Query": query,
        "General Model": hit_gen['score'],
        "Medical Model": hit_med['score']
    })


df_res = pd.DataFrame(results)


plt.figure(figsize=(10, 5))

df_melted = df_res.melt(id_vars="Query", var_name="Model", value_name="Confidence Score")

sns.barplot(data=df_melted, x="Query", y="Confidence Score", hue="Model", palette=["gray", "green"])
plt.ylim(0.5, 1.0) # Zoom in on the top half of the graph
plt.title("Confidence Comparison: General vs Medical AI")
plt.ylabel("Confidence Score (Higher is Better)")
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()


display(df_res)

In [ ]:
from sentence_transformers import util
import pandas as pd


target_query = "Treatment for pulmonary hypertension"


query_emb_gen = model_general.encode(target_query, convert_to_tensor=True)
query_emb_med = model_medical.encode(target_query, convert_to_tensor=True)


hit_gen = util.semantic_search(query_emb_gen, embeddings_general, top_k=1)[0][0]
hit_med = util.semantic_search(query_emb_med, embeddings_medical, top_k=1)[0][0]


retrieved_q_gen = df_sample.iloc[hit_gen['corpus_id']]['question']
score_gen = hit_gen['score']


retrieved_q_med = df_sample.iloc[hit_med['corpus_id']]['question']
score_med = hit_med['score']


results_table = pd.DataFrame({
    "Model": ["Model 1: General (MiniLM)", "Model 2: Medical (PubMedBERT)"],
    "Your Query": [target_query, target_query],
    "Retrieved Database Question": [retrieved_q_gen, retrieved_q_med],
    "Confidence Score": [score_gen, score_med]
})


print("\n--- RESULTS FOR TABLE 7.1 ---")
display(results_table)


print(f"\nExact Values for your Report:")
print(f"General Score: {score_gen:.4f}")
print(f"Medical Score: {score_med:.4f}")